# House Price Prediction Project

**Lucy, Elliott, Carter, George, Connor**

In this project, we will build a machine learning model to predict housing prices in the United States using property-related data from the USA Housing Dataset on Kaggle.

Housing prices are influenced by a variety of factors, including the size of the home, number of rooms, condition, and location. The goal of this project is to use these features to train a model that can accurately estimate the price of a house.

This is a **supervised learning regression problem**, where:
- The **input features** describe characteristics of a house
- The **target variable** is the house price

The ultimate goal is to develop a model that can generalize well to new data and provide accurate and meaningful predictions of house prices.

## Data Loading

In [ ]:

# Import required libraries
import pandas as pd
import numpy as np
import glob
import os

import matplotlib.pyplot as plt
import seaborn as sns

# Settings
np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Load the dataset
csv_file = "USA Housing Dataset.csv"

if len(csv_file) == 0:
    raise FileNotFoundError("No CSV files were found in the 'usa_house_prices_dataset' folder.")

print("Loaded file:", csv_file)

df = pd.read_csv(csv_file)

Loaded file: USA Housing Dataset.csv


FileNotFoundError: [Errno 2] No such file or directory: 'USA Housing Dataset.csv'

## Task 1: Understand the Dataset

The dataset used for this project is the **USA Housing Dataset** from Kaggle. Each row represents one house sale, and each column gives information about the property, such as its size, location, condition, and sale price.

The goal is to understand the structure of the dataset before building a machine learning model.

For this task, we will identify:
1. The features used to describe each house
2. The target variable the model will predict
3. The number of samples in the dataset

In [ ]:
# Display the first few rows
df.head()

In [ ]:
# Basic dataset information
print("Dataset shape:", df.shape)
print("Number of samples:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
# Identify the target variable
target = "price"

# Identify the features
features = df.drop(columns=[target]).columns.tolist()

print("Target variable:", target)
print("\nFeatures:")
for feature in features:
    print("-", feature)

In [ ]:
# Check data types
df.info()

### Task 1 Summary

The dataset contains **4,140 samples**, meaning there are 4,140 house sale records.

The **target variable** is `price`, which represents the sale price of the house. Since price is a continuous numerical value, this project is a **supervised regression problem**.

The **features** are the remaining columns used to predict price. These include numerical property characteristics such as bedrooms, bathrooms, square footage, lot size, number of floors, condition, year built, and year renovated. The dataset also includes location-related categorical features such as street, city, statezip, and country.

Before modeling, the data must be cleaned and converted into a numerical format so that it can be used by a machine learning algorithm.

## Task 2: Data Preprocessing

Before training the model, the dataset needs to be prepared for machine learning.

This preprocessing step includes:
1. Handling missing values
2. Converting categorical variables into numerical values
3. Normalizing/scaling numerical data
4. Splitting the dataset into training and testing sets

These steps help ensure that the model can process the data correctly and make fair predictions.

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
print("\nTotal missing values:", missing_values.sum())

In [ ]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print("Number of duplicate rows:", duplicates)

In [ ]:
# Convert date column into useful numeric features
df["date"] = pd.to_datetime(df["date"], errors="coerce")

df["sale_year"] = df["date"].dt.year
df["sale_month"] = df["date"].dt.month

# Drop original date column after extracting useful parts
df = df.drop(columns=["date"])

In [ ]:
# Drop columns that are too specific or not useful for prediction
# Street addresses are high-cardinality and may cause overfitting
df = df.drop(columns=["street", "country"])

print("Remaining columns:")
print(df.columns.tolist())

In [ ]:
# Separate features and target variable
X = df.drop(columns=["price"])
y = df["price"]

print("Feature data shape:", X.shape)
print("Target data shape:", y.shape)

In [ ]:
# Split data into training and testing sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

In [ ]:
# Identify numerical and categorical columns
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

In [ ]:
# Build preprocessing pipeline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# Numerical columns: fill missing values with median, then scale
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical columns: fill missing values with most frequent value, then one-hot encode
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both preprocessing steps
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [ ]:
# Fit the preprocessor on the training data only
# Then transform both training and testing data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training data shape:", X_train_processed.shape)
print("Processed testing data shape:", X_test_processed.shape)

### Task 2 Summary

The dataset was preprocessed so it could be used in a machine learning model.

First, missing values were checked. Even though this dataset does not appear to have missing values, imputation was still included in the preprocessing pipeline to make the workflow more reliable.

Duplicate rows were also checked and removed if present.

The `date` column was converted into two more useful numerical features: `sale_year` and `sale_month`. The original date column was then removed because machine learning models cannot directly use raw date strings.

The `street` column was removed because individual street addresses are too specific and could cause the model to memorize locations instead of learning general patterns. The `country` column was also removed because all rows are from the USA, so it does not add useful information.

The data was then split into:
- **80% training data**
- **20% testing data**

Finally, numerical features were scaled using `StandardScaler`, and categorical features such as `city` and `statezip` were converted into numerical columns using one-hot encoding.

## Task 3: Build a Model

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def evaluate_regression(y_true, y_pred, model_name):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"--- {model_name} ---")
    print(f"MSE:  {mse:,.3f}")
    print(f"RMSE: {rmse:,.3f}")
    print(f"MAE:  {mae:,.3f}")
    print(f"R²:   {r2:.3f}")
    print()

    return {"Model": model_name, "MSE": mse, "RMSE": rmse, "MAE": mae, "R2": r2}

## Task 4: Train and Test

In [ ]:
linreg = LinearRegression()
linreg.fit(X_train_processed, y_train)

y_test_pred_linreg = linreg.predict(X_test_processed)
linreg_results = evaluate_regression(y_test, y_test_pred_linreg, "Linear Regression")

y_test_pred = linreg.predict(X_test_processed)

## Task 5: Evaluate Results

In [ ]:
test_results = evaluate_regression(y_test, y_test_pred, "Final Model (Test Set)")

## Task 6: Visualization

### Plot predicted vs. actual prices

In [ ]:
plt.figure(figsize=(7, 7))
sns.scatterplot(x=y_test, y=y_test_pred, alpha=0.6)

min_val = min(y_test.min(), y_test_pred.min())
max_val = max(y_test.max(), y_test_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")

plt.title("Predicted vs. Actual House Prices")
plt.xlabel("Actual price")
plt.ylabel("Predicted price")
plt.show()

### Plot residuals

In [ ]:
residuals = y_test - y_test_pred

sns.histplot(residuals, bins=40)
plt.title("Residual Distribution (Actual - Predicted)")
plt.xlabel("Residual")
plt.ylabel("Count")
plt.show()

### View the largest coefficients by absolute value

In [ ]:
feature_names = preprocessor.get_feature_names_out()
coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": linreg.coef_
})
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coefficient", ascending=False)

coef_df.head(15)

### Plot the top coefficients

In [ ]:
top_coefs = coef_df.head(15).sort_values("coefficient")

sns.barplot(data=top_coefs, x="coefficient", y="feature")
plt.title("Top 15 Linear Regression Coefficients (by absolute value)")
plt.xlabel("Coefficient")
plt.ylabel("Feature")
plt.show()